# 4.2 De lange staart, toegepast op taxiritten

[04.1](04.1-parametric-distributions%20jeffrey.ipynb) tekende zes verdelingsfamilies uit bekende parameters en benoemde het mechanisme achter elke vorm. Dit notebook past er één toe op echte data. De vraag is niet simpelweg "hoe ziet een ritprijs eruit" — maar *welke familie past, en waarom*, omdat het antwoord bepaalt wat telt als een uitzonderlijk dure rit. We bouwen voort op de mechanismen uit 04.1, in het bijzonder de centrale limietstelling (een product wordt een som onder de logaritme), en de goad-onderdelen die in 04.1 zijn geïntroduceerd: `DistributionFitter`, `fit_table` met haar twee winnaars, `PlotFits` en `QQPlot`.

`taxis`: 6.433 taxiritten in New York City. Denk eerst na over het onderliggende mechanisme. Een ritprijs bestaat uit een basistarief plus een meter die oploopt op basis van afstand *én* tijd vast in het verkeer, aangevuld met tolgelden en toeslagen bij sommige ritten. De afstand zelf is scheef verdeeld — de meeste ritten zijn kort, enkele doorkruisen de hele stad — en langzaam verkeer vermenigvuldigt de kosten van elke extra kilometer. Meerdere factoren die elkaar grotendeels *vermenigvuldigen*, met een harde ondergrens bij het basistarief en geen bovengrens: dat is exact het productmechanisme uit 04.1 §4.1.2, en dat voorspelt een lognormale verdeling. Toets deze voorspelling in drie stappen: de samenvatting die de staart verraadt, de visuele vergelijking met en zonder logaritme, en vervolgens de fitter.


In [ ]:
import numpy as np

from goad_toolkit.analytics import DistributionFitter, fit_table
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.visualizer import FitPlotSettings, HistogramPlot, PlotFits, PlotSettings, QQPlot

from wa_analyzer.data import load_showcase

## 4.2.1 De samenvatting die de staart verraadt


In [ ]:
taxis = load_showcase("taxis")
fare = taxis["fare"].to_numpy()

print(f"mean:   {fare.mean():.2f}")
print(f"median: {np.median(fare):.2f}")

Het gemiddelde ligt aanzienlijk boven de mediaan — de lange staart trekt het gemiddelde omhoog. Bij een symmetrische verdeling zouden beide samenvallen; dit verschil is dus het eerste signaal dat "gemiddelde ± standaarddeviatie" een ritprijs zou beschrijven die in werkelijkheid niet bestaat. Nu de grafiek, twee keer: ruw, en na het nemen van de logaritme. **De logaritme is de oplossing**: als het mechanisme multiplicatief is, is `log(fare)` een *som* van de gelogde factoren, en volgens 04.1 §4.1.2 zou een som er normaal verdeeld uit moeten zien.


In [ ]:
tail = PlotSettings(
    figsize=(10, 3.5),  # ty: ignore[invalid-argument-type]
    title="The same fares, before and after the log",
    subplot_titles=["fare: a long right tail", "log(fare): the tail becomes a bell"],
    max_cols=2,
)
host = HistogramPlot(tail)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(HistogramPlot(tail), axes[0], data=fare, color="steelblue")
_ = host.plot_on_axes(HistogramPlot(tail), axes[1], data=np.log(fare), color="steelblue")

## 4.2.2 Laat de fitter het oordeel vellen

De grafiek ondersteunt de voorspelling; de fitter kwantificeert het in een getal. `fit()` op de ruwe ritprijzen probeert elke continue familie in het register en rangschikt ze. Lees de tabel op twee cruciale zaken: wint `lognorm`, en met hoeveel verslaat het `norm` — de verdeling die een standaard z-score-regel stilzwijgend veronderstelt?


In [ ]:
fitter = DistributionFitter(DistributionRegistry(), seed=42)
fare_fits = fitter.fit(fare, discrete=False)
fit_table(fare_fits)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]]

In [ ]:
fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="fare", ylabel="density", title="Taxi fares: the top three fits")).plot(
    data=fare, fit_results=fare_fits, fitplotsettings=FitPlotSettings(bins=40, max_fits=3),
)

## 4.2.3 De staart, waar families van elkaar verschillen

Een histogram met een curve eroverheen wordt gedomineerd door de bulk van de data — precies waar verschillende families het meest op elkaar lijken. De staart is waar ze werkelijk van elkaar afwijken, en de staart is exact wat een `QQPlot` toont: gesorteerde ritprijzen afgezet tegen de kwantielen die een gefitte familie voorspelt, met een referentielijn. Punten die afbuigen van de lijn zijn waarden die de familie niet kan verklaren.


In [ ]:
norm_fit = fitter.fit_distribution("norm", fare)
lognorm_fit = fitter.fit_distribution("lognorm", fare)

qq = PlotSettings(
    figsize=(10, 4.5),  # ty: ignore[invalid-argument-type]
    title="Which family explains the expensive rides?",
    subplot_titles=["fare vs. fitted normal", "fare vs. fitted lognormal"],
    xlabel="theoretical quantile",
    ylabel="fare",
    max_cols=2,
)
host = QQPlot(qq)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(QQPlot(qq), axes[0], data=fare, distribution=norm_fit.frozen_dist)  # ty: ignore[unresolved-attribute]
_ = host.plot_on_axes(QQPlot(qq), axes[1], data=fare, distribution=lognorm_fit.frozen_dist)  # ty: ignore[unresolved-attribute]

De normale qq-plot buigt in de rechterbovenstaart scherp af van de referentielijn — precies waar een histogram het minst afleesbaar is, en exact waar een "z-score boven de 3"-regel gewone dure ritten ten onrechte als anomalieën zou bestempelen. De lognormale verdeling volgt de referentielijn daarentegen veel verder naar buiten. Zelfde data; het verschil zit puur in wat elke familie verwacht dat de staart zou moeten doen.

## 4.2.4 De allerduurste taxirit

Wat dit concreet betekent voor de allerduurste rit in de dataset:


In [ ]:
worst_fare = fare.max()
z = (worst_fare - fare.mean()) / fare.std()
tail_prob = 1 - lognorm_fit.frozen_dist.cdf(worst_fare)  # ty: ignore[unresolved-attribute]
print(f"most expensive fare: ${worst_fare:.2f}, n={len(fare):,} rides")
print(f"z-score under a normal fit: {z:.1f}  (a real normal puts this at ~1-in-10^32)")
print(f"P(fare >= this) under the fitted lognormal: {tail_prob:.1e}  (~1-in-{1/tail_prob:,.0f})")

Onder de lognormale fit is deze rit nog steeds zeldzaam — 1 op de 19.000 is zeker niet "alledaags" — maar de twee modellen verschillen ongeveer 27 ordes van grootte over *hóe* zeldzaam. Het normale model noemt deze rit nagenoeg mathematisch onmogelijk; het lognormale model beschouwt het als een zeldzame, maar reële trekking uit een staart waar elke lange rit aan blootstaat. Geen van beide modellen zegt "verwijder deze rij"; alleen het verkeerde model beweert "dit kan niet gebeurd zijn." Dat is reden 2 uit [04.1](04.1-parametric-distributions%20jeffrey.ipynb): een uitbijter is een uitspraak over een verdeling, en die uitspraak is slechts zo goed als de familie die erachter schuilgaat.


---

**Waar dit nu naartoe gaat.** Hier waren de fitter en het mechanisme het met elkaar eens: één proces, één familie, één staart. [04.3-penguins](04.3-penguins%20jeffrey.ipynb) behandelt het geval waarin de fitter het met geen enkele verdeling eens kan worden — elke familie wordt verworpen. De reden is geen vreemde verdelingsvorm, maar een populatie die in werkelijkheid uit meerdere groepen bestaat. Het gereedschap blijft hetzelfde; wat verandert is de vraag die je stelt wanneer de fit faalt.
